# Preprocessing CICIDS2017 Dataset

In [4]:
#use this only if you are doing the code in kaggle or else ignore it.
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Tuesday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Monday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv
/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Wednesday-workingHours.pcap_ISCX.csv


In [5]:
import pandas as pd

In [7]:
dataset1 = pd.read_csv("/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset/Tuesday-WorkingHours.pcap_ISCX.csv")
dataset1.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,88,640,7,4,440,358,220,0,62.857143,107.349008,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,88,900,9,4,600,2944,300,0,66.666667,132.287566,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1205,7,4,2776,2830,1388,0,396.571429,677.274651,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,88,511,7,4,452,370,226,0,64.571429,110.276708,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,773,9,4,612,2944,306,0,68.000000,134.933317,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
RAW_DIR = "/kaggle/input/datasets/sateeshkumar6289/cicids-2017-dataset"
OUT_DIR = "/kaggle/working/processed" #this is the directory where we will be storing our preprocessed final data


In [7]:
#you will get a combined, one dataset finally, for that you create a directory and write the dataset onto it. 
os.makedirs(OUT_DIR, exist_ok=True)

In [8]:
# STEP 1: Load and combine the CSV files
csv_paths = glob.glob(os.path.join(RAW_DIR, "*.csv"))
df_list = [pd.read_csv(path, low_memory=False, encoding="latin1") for path in csv_paths]
df = pd.concat(df_list)
df = df.reset_index(drop=True)
print("Combined shape:", df.shape)

Combined shape: (2830743, 79)


In [9]:
# STEP 2: Clean column names (strip stray whitespace)
df.columns = df.columns.str.strip()

In [10]:
# STEP 3: Identify the label column
label_col = "Label"
print("Label column:", label_col)
print(df[label_col].value_counts())


Label column: Label
Label
BENIGN                          2273097
DoS Hulk                         231073
PortScan                         158930
DDoS                             128027
DoS GoldenEye                     10293
FTP-Patator                        7938
SSH-Patator                        5897
DoS slowloris                      5796
DoS Slowhttptest                   5499
Bot                                1966
Web Attack ï¿½ Brute Force         1507
Web Attack ï¿½ XSS                  652
Infiltration                         36
Web Attack ï¿½ Sql Injection         21
Heartbleed                           11
Name: count, dtype: int64


In [11]:
# STEP 4: Remove duplicate rows (before the split, to avoid the leakage)
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after dedup:", df.shape)

Duplicates: 308381
Shape after dedup: (2522362, 79)


In [12]:
# STEP 5: Handle infinite and missing values
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print("Rows with missing values:", df[numeric_cols].isna().any(axis=1).sum())
df = df.dropna(subset=numeric_cols)
print("Shape after dropping missing rows:", df.shape)

Rows with missing values: 1564
Shape after dropping missing rows: (2520798, 79)


In [13]:
# STEP 6: Drop identifier / leakage-risk columns
id_cols = ["Flow ID", "Source IP", "Src IP", "Destination IP", "Dst IP", "Timestamp"]
df = df.drop(columns=[c for c in id_cols if c in df.columns])

In [14]:
# STEP 7: Drop constant (zero-variance) columns
numeric_cols = df.select_dtypes(include=[np.number]).columns  # recompute after drops
constant_cols = [c for c in numeric_cols if df[c].nunique() <= 1]
df = df.drop(columns=constant_cols)
print("Dropped constant columns:", constant_cols)

Dropped constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [15]:
# STEP 8: Build the labels
df[label_col] = df[label_col].astype(str).str.strip()
df["Binary_Label"] = (df[label_col] != "BENIGN").astype(int)
df["Attack_Category"] = df[label_col]
df = df.drop(columns=[label_col])
print("\nClass distribution:")
print(df["Attack_Category"].value_counts())



Class distribution:
Attack_Category
BENIGN                          2095057
DoS Hulk                         172846
DDoS                             128014
PortScan                          90694
DoS GoldenEye                     10286
FTP-Patator                        5931
DoS slowloris                      5385
DoS Slowhttptest                   5228
SSH-Patator                        3219
Bot                                1948
Web Attack ï¿½ Brute Force         1470
Web Attack ï¿½ XSS                  652
Infiltration                         36
Web Attack ï¿½ Sql Injection         21
Heartbleed                           11
Name: count, dtype: int64


In [16]:
# STEP 9: Separate features/labels, then train/test split
feature_cols = [c for c in df.columns if c not in ("Binary_Label", "Attack_Category")]
X = df[feature_cols]
y = df["Binary_Label"]

X_train, X_test, y_train, y_test, cat_train, cat_test = train_test_split(
    X, y, df["Attack_Category"],
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])


Train rows: 2016638 | Test rows: 504160


In [17]:
# STEP 10: Scale features (fit on train only, apply to test)
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

In [18]:

# STEP 11: Save processed data
X_train_scaled["Binary_Label"] = y_train.values
X_train_scaled["Attack_Category"] = cat_train.values

X_test_scaled["Binary_Label"] = y_test.values
X_test_scaled["Attack_Category"] = cat_test.values

X_train_scaled.to_csv(os.path.join(OUT_DIR, "train1.csv"), index=False)
X_test_scaled.to_csv(os.path.join(OUT_DIR, "test1.csv"), index=False)

#If you are using any other IDE(not kaggle), checke in which folder you wish to store the output.
print("\nSaved train1.csv and test1.csv to", OUT_DIR)


Saved train.csv and test.csv to /kaggle/working/processed
